Prepare data, build model

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, layers=3, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.layers = [nn.Linear(input_size, hidden_size),
                       nn.ReLU()]
        for _ in range(layers - 1):
            self.layers += [nn.Linear(hidden_size, hidden_size),
                            nn.ReLU()]
        self.layers += [nn.Linear(hidden_size, output_size)]
        self.model = nn.Sequential(*self.layers)
    
    def forward(self, x):
        x = x.flatten(start_dim=1) # flatten the 28x28 images to a vector of size 784
        return self.model(x)

# dataset gets downloaded to data/ and transformed to tensors
# MNIST is a basic handwritten digit dataset with 10 classes (0-9) and 28x28 pixel greyscale images
train_ds = datasets.MNIST(root='data', train=True, download=True, transform=transforms.ToTensor())
test_ds = datasets.MNIST(root='data', train=False, download=True, transform=transforms.ToTensor())

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=128)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNN(28*28, 128, 10).to(device)


prepare training settings

In [2]:
model.train()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10


run training

In [ ]:

# for tracking training history
train_loss_history = []
train_acc_history = []

for epoch in range(num_epochs):
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in train_dl:
        # prepare data
        images, labels = images.to(device), labels.to(device)
        images = images.view(-1, 28*28)

        # forward + backward + optimize
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # collect statistics
        running_loss += loss.item() * images.size(0)
        predicted = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    train_loss_history.append(epoch_loss)
    train_acc_history.append(epoch_acc)
    print(f'Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss:.4f} | Train-Accuracy: {epoch_acc:.4f}')


basic evaluation

In [ ]:
test_preds = []
test_labels = []
test_data = []
model.eval()
with torch.no_grad(): # this suppresses gradient computation to save memory when we dont need the gradients
    for images, labels in test_dl:
        # Add code to evaluate the model on the test set and compute test accuracy
        # handle the data and collect the statistics for test accuracy
        # you need the whole test set later
        ...

test_preds = ...
test_labels = ...
test_data = ...
test_acc = ...
print(f'Test Accuracy: {test_acc:.4f}')

visualization of training history

In [ ]:
# make a plot of the training history
# include both loss and accuracy in the same plot
# make two y-axis in the plot, one for loss and one for accuracy
# add descriptions and legends to the plot

...

plt.show()

basic adversarial attack

In [ ]:
# basic adversarial attack that changes the pixels of a given input image in the direction of the gradient of the loss with respect to the input
# This way the image is changes so that the model is more likely to make a wrong prediction
# Epsilon controls how much the image is changed, a higher epsilon means a stronger attack
class Fast_Gradient_Sign_Method(nn.Module):
    def __init__(self, model, epsilon=0.1, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.model = model
        self.epsilon = epsilon
    
    def forward(self, x, y):
        # make the input require gradients
        x.requires_grad = True

        # forward pass
        outputs = self.model(x)
        loss = criterion(outputs, y)

        # backward pass
        self.model.zero_grad()
        loss.backward()

        # collect the sign of the gradients
        x_grad_sign = x.grad.sign()

        # create the adversarial example by adding the perturbation
        x_adv = x + self.epsilon * x_grad_sign

        return x_adv

run attack

In [ ]:
fgsm = Fast_Gradient_Sign_Method(model, epsilon=0.1)
# create adversarial examples for the test set

test_adv = fgsm(test_data[:50], test_labels[:50])

with torch.no_grad():
    adv_predictions = model(test_adv)
    adv_pred_labels = adv_predictions.argmax(dim=1)

adv_acc = (adv_pred_labels == test_labels[:50]).sum().item() / 50
print(f'Adversarial Test Accuracy: {adv_acc:.4f}')


In [ ]:
from torchvision.utils import make_grid

# visualize some of the adversarial examples
grid = make_grid(test_adv.cpu(), nrow=10, normalize=True)
plt.figure(figsize=(10, 10))
plt.imshow(grid.permute(1, 2, 0))
plt.title('Adversarial Examples')

adjust attack

In [ ]:
# right now the whole test dataset is processed by the attack in one call, that is too big for many machines, so we need to process the test set in batches
# complete the altered attack-class so that it can process the test set in batches and outputs the combined data

class Fast_Gradient_Sign_Method_Batch(nn.Module):
    def __init__(self, model, epsilon=0.1, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.model = model
        self.epsilon = epsilon
    
    def forward(self, testloader):
        # this method gets the whole dataset testloader
        # every batch should be processed individually and after that they should be combined to one tensor of adversarial examples 
        # the resulting tensor should be returned
        test_adv_combined = ...
        
        ...

        return test_adv_combined


In [ ]:
# the adjusted attack should work like this:

fsgm_batch = Fast_Gradient_Sign_Method_Batch(model, epsilon=0.1)
test_adv_batch = fsgm_batch(test_dl) # results in one big tensor, if you use the whole MNIST-testset, it should have the dimension [10000 x 28*28]

from torch.utils.data import TensorDataset
with torch.no_grad():
    adv_dataset = TensorDataset(test_adv_batch, test_labels)
    adv_dl = DataLoader(adv_dataset, batch_size=128)

    adv_predictions = []
    for adv_images, adv_labels in adv_dl:
        outputs = model(adv_images)
        adv_predictions.append(outputs)

adv_predictions = torch.cat(adv_predictions, dim=0)
adv_pred_labels = adv_predictions.argmax(dim=1)

adv_acc = (adv_pred_labels == test_labels).sum().item() / len(test_labels)
print(f'Adversarial Test Accuracy: {adv_acc:.4f}')


You can try to experiment with the values of epsilon to see what changes are made to the data. You can experiment with different values and look at the resulting images and accuracies.

In [ ]:
# Extra Task:
# To understand the influence of the epsilon parameter on the attack, run it with different values of epsilon [0.01, 0.05, 0.1, 0.15, ...].
# Store the resulting accuracies and plot what accuracy relates to what epsilon value.
epsilons = [0.01, 0.05, 0.1, 0.15] # add more values
adv_accuracies = []
for eps in epsilons:
    # run attack
    ...
    adv_acc = ...
    adv_accuracies.append(adv_acc)

# plot the results
...

the resulting plot should look like this:

![Epsilon curve](epsilon_sketch.png)